In [9]:
# !pip install torchvision, pillow

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [2]:
class ImageProcessor:
    def __init__(self, dir_path, transformations=None):
        self.dir_path = dir_path,
        self.transformations = transformations

        self.all_img_pth = [os.path.join(dir_path, img) for img in os.listdir(dir_path)]
    def __len__(self):
        return len(self.all_img_pth)

    def __getitem__(self, idx):
        img_path = self.all_img_pth[idx]
        img = Image.open(img_path).convert('RGB')

        if self.transformations:
            img = self.transformations(img)

        return img

In [3]:
dir_path = "X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\GAN\\img_align_celeba"
transformations = transforms.Compose(
    [
        transforms.CenterCrop(178),
        transforms.Resize(64),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

In [4]:
dataset = ImageProcessor(dir_path, transformations)
print(f"Loaded {len(dataset)} Images")

Loaded 202599 Images


In [5]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

In [6]:
# Generator Network

In [7]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=3):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            
            nn.Linear(256, 512),
            nn.ReLU(),
            
            nn.Linear(512, 1024),
            nn.ReLU(),
            
            nn.Linear(1024, 64 * 64 * img_channels),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0),3, 64, 64)
        return img

In [8]:
# Discriminator

In [9]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),
            
            nn.Linear(img_channels*64*64, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(256, 1),
            nn.Sigmoid()

        )

    def forward(self, img):
        return self.model(img)


In [10]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

discrimitor = Discriminator()
d_optimizer = optim.Adam(discrimitor.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [11]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device is {device}")

Device is cuda


In [12]:
generator = generator.to(device)
discrimitor = discrimitor.to(device)

In [13]:
# Train GAN

def train(generator, discrimitor, dataloader, epochs=10):

    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100).to(device))

            real_loss = GAN_loss(discrimitor(real_imgs), real_labels)
            fake_loss = GAN_loss(discrimitor(fake_imgs.detach()), fake_labels)


            d_loss = (real_loss + fake_loss)/2
            d_loss.backward()
            d_optimizer.step()

            g_optimizer.zero_grad()
            g_loss = GAN_loss(discrimitor(fake_imgs), real_labels)
            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"For Epoch:  {epoch+1}/{epochs}\n, Batch: {i+1}, G-Loss: {g_loss}, D-Loss: {d_loss}")


        save_generated_images(generator, epoch, device)

In [14]:
def save_generated_images(generator, epoch, device, num_img=8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_img = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_img, nrow=4, normalize=True)

    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis('off')
    plt.show()

In [ ]:
train(generator, discrimitor, dataloader, epochs=5)

For Epoch:  1/5
, Batch: 1, G-Loss: 0.7015976309776306, D-Loss: 0.6888089179992676
For Epoch:  1/5
, Batch: 51, G-Loss: 1.07261323928833, D-Loss: 0.2553453743457794
For Epoch:  1/5
, Batch: 101, G-Loss: 2.1400675773620605, D-Loss: 0.11430229991674423
For Epoch:  1/5
, Batch: 151, G-Loss: 1.7662439346313477, D-Loss: 0.20823654532432556
For Epoch:  1/5
, Batch: 201, G-Loss: 1.384232521057129, D-Loss: 0.2305036038160324
For Epoch:  1/5
, Batch: 251, G-Loss: 1.0627782344818115, D-Loss: 0.33196887373924255
For Epoch:  1/5
, Batch: 301, G-Loss: 1.8657262325286865, D-Loss: 0.10840871185064316
For Epoch:  1/5
, Batch: 351, G-Loss: 1.7499760389328003, D-Loss: 0.1854582130908966
For Epoch:  1/5
, Batch: 401, G-Loss: 2.591583251953125, D-Loss: 0.17728020250797272
For Epoch:  1/5
, Batch: 451, G-Loss: 2.816903591156006, D-Loss: 0.14696572721004486
For Epoch:  1/5
, Batch: 501, G-Loss: 2.5919113159179688, D-Loss: 0.13134083151817322
For Epoch:  1/5
, Batch: 551, G-Loss: 3.4315590858459473, D-Loss: 